# Variable selection — memory profiling

Takes a **sample** of the raw Home & Kitchen metadata and measures how much
memory each variable costs, so we can tell which variables are worth carrying
and which to exclude.

Two passes:

1. **Profile** — load a 100k-row sample of the raw JSON and rank every field by
   its deep memory footprint.
2. **Trim and re-measure** — reload keeping only the fields worth having, save
   the result to `data/meta_Home_and_Kitchen_filtered.csv`, and re-measure to
   confirm the saving.

Excluded on the way: `similar_item`, `also_buy`, `also_view`, `fit`, and
`details` (mostly missing).

> ⚠️ The save cell writes `data/meta_Home_and_Kitchen_filtered.csv` (~2.1 GB),
> which is the input to `feature_extraction_workflow/`. Running it overwrites
> the existing file.


## From the metadata, find variables using the most memory

In [1]:
# Randomly pick the first 1,000 rows
import json
from pathlib import Path

import pandas as pd

# This notebook lives in <repo>/data/, so the project's data folder is the
# notebook's own directory; fall back to <cwd>/data if the kernel starts at the
# repo root. Every output below is written here.
DATA_DIR = Path.cwd() if Path.cwd().name == "data" else Path.cwd() / "data"

file = '/Users/lazr/Desktop/Rec Engine/Datasets/meta_Home_and_Kitchen.json'

selected_values = []
with open(file, 'r') as fp:
    for i, line in enumerate(fp):
        if i >= 100000:
            break
        selected_values.append(json.loads(line.strip()))


In [2]:
df = pd.DataFrame(selected_values)

In [3]:
df.head()

,category,tech1,description,fit,title,also_buy,tech2,brand,feature,rank,also_view,main_cat,similar_item,date,price,asin,imageURL,imageURLHighRes,details
0,"[Home & Kitchen, Kitchen & Dining, Dining & En...",,[It was a time honored tradition among the ear...,,You Are Special Today Red Plate [With Red Pen],"[B0001XR2F2, B01LY51HUN, B07CXZ9C5B, 0310258952]",,Waechtersbach USA,[],"[>#39,665 in Kitchen & Dining (See Top 100 in ...","[B0001XR2F2, B00MOFKX1A, B07G3LN13B, B07CYXMFF...",Amazon Home,"class=""a-bordered a-horizontal-stripes a-spa...","October 8, 2006",$37.00,0001487795,[],[],NaN
1,"[Home & Kitchen, Home Dcor, Candles & Holders,...",,[VICKS INHALER relieves stuffy noses helps sin...,,Vicks Inhaler Relief for Cold Sinus Nasal Cong...,[],,Vicks,[],"[>#1,763,185 in Home & Kitchen (See Top 100 in...",[B00UPCRZEC],Amazon Home,,,$4.05,0002020300,[],[],NaN
2,"[Home & Kitchen, Kitchen & Dining, Dining & En...",,"[16 oz squeeze bottle, 1 lb.]",,Artistic Churchware Communion Cup Filler: RW525,[],,Artistic Churchware,"[Religious Supply Center, RW-525, Communion Cu...","[>#2,127,003 in Home & Kitchen (See Top 100 in...","[B00C9J79TA, B00XB3ZG5M, B0714K8MSR, 078472530...",Amazon Home,,,$12.48,0006564224,[],[],NaN
3,"[Home & Kitchen, Bath, Bathroom Accessories]",,[The only soap in the world with a unique comb...,,4 BARS! Mysore Sandal Soap 70grams FAST SHIPPING,[],,Mysore,[],"[>#6,942,841 in Home & Kitchen (See Top 100 in...",[B001G7PZB0],Amazon Home,,,$22.00,0009046461,[https://images-na.ssl-images-amazon.com/image...,[https://images-na.ssl-images-amazon.com/image...,NaN
4,"[Home & Kitchen, Home Dcor, Home Fragrance, In...",,"[Divya Arogya vati improves health, It has nat...",,AROGYA VATI (40gm) by popeye seller,[],,Patanjali,[],"[>#3,103,399 in Home & Kitchen (See Top 100 in...","[B075M91MX3, 0258798785, 6565652554]",Amazon Home,,,$5.10,0234937912,[],[],NaN


In [4]:
# Calculate memory usage in Megabytes
usage = df.memory_usage(deep=True) / (1024 ** 2)

# Sort and display
usage_df = usage.sort_values(ascending=False).to_frame(name='MB')
print(usage_df)

                         MB
similar_item     952.840597
tech1             25.031378
price             19.694850
also_view         12.553101
category          10.594543
feature           10.516571
title              9.956681
also_buy           9.224182
description        9.052582
rank               9.037814
details            7.432663
imageURL           7.313934
imageURLHighRes    7.313934
main_cat           5.842399
date               5.717591
brand              5.692658
asin               5.626678
fit                4.703371
tech2              4.673004
Index              0.000126


In [5]:
usage_df['MB'].sum()

np.float64(1122.8186588287354)

## **From metadata, take only the necessary information**

In [6]:
# Show full column width (no truncation)
pd.set_option('display.max_colwidth', 500)

In [7]:
import json

file = '/Users/lazr/Desktop/Rec Engine/Datasets/meta_Home_and_Kitchen.json'

# Fields you want to keep (replace or remove/add as needed)
# Fields removed "similar_item", "also_buy", "also_view", "details", "fit"
# "details" is dropped because it is mostly missing
fields_to_keep = ["category", "tech1", "description", "title", "tech2", "brand", "feature", "rank", "main_cat", "price", "asin", "date", "imageURL", "imageURLHighRes"]

data = []

with open(file, 'r') as fp:
    for i, line in enumerate(fp):
        # if i >= 5000000:  # stop after 1000 rows
            # break
        obj = json.loads(line.strip())
        # keep only the needed fields
        filtered_obj = {k: obj[k] for k in fields_to_keep if k in obj}
        data.append(filtered_obj)

# Convert to DataFrame
df = pd.DataFrame(data)


In [8]:
# Save into this project's data/ folder
OUT_PATH = DATA_DIR / 'meta_Home_and_Kitchen_filtered.csv'

df.to_csv(OUT_PATH, index=False)
print(f'saved {len(df):,} rows x {df.shape[1]} cols -> {OUT_PATH.resolve()}')


In [9]:
df.head()

,category,tech1,description,title,tech2,brand,feature,rank,main_cat,price,asin,date,imageURL,imageURLHighRes
0,"[Home & Kitchen, Kitchen & Dining, Dining & Entertaining, Dinnerware, Plates, Dinner Plates]",,[It was a time honored tradition among the early American families that when someone deserved special praise or attention they were served dinner on the Red Plate. Observe the special occasions of each],You Are Special Today Red Plate [With Red Pen],,Waechtersbach USA,[],"[>#39,665 in Kitchen & Dining (See Top 100 in Kitchen & Dining), >#93 in Kitchen & Dining > Tabletop > Dinnerware > Plates > Dinner Plates, >#19,436 in Home & Kitchen > Kitchen & Dining > Food Service Equipment & Supplies]",Amazon Home,$37.00,0001487795,"October 8, 2006",[],[]
1,"[Home & Kitchen, Home Dcor, Candles & Holders, Candles]",,[VICKS INHALER relieves stuffy noses helps sinus congestion breathe easy great for allergy season made in India],Vicks Inhaler Relief for Cold Sinus Nasal Congestion Allergy,,Vicks,[],"[>#1,763,185 in Home & Kitchen (See Top 100 in Home & Kitchen), >#29,437 in Home & Kitchen > Home Dcor > Candles & Holders > Candles]",Amazon Home,$4.05,0002020300,,[],[]
2,"[Home & Kitchen, Kitchen & Dining, Dining & Entertaining, Glassware & Drinkware, Wine & Champagne Glasses]",,"[16 oz squeeze bottle, 1 lb.]",Artistic Churchware Communion Cup Filler: RW525,,Artistic Churchware,"[Religious Supply Center, RW-525, Communion Cup Filler]","[>#2,127,003 in Home & Kitchen (See Top 100 in Home & Kitchen), >#11,867 in Home & Kitchen > Kitchen & Dining > Tabletop > Bar Tools & Glasses > Wine & Champagne Glasses, >#138,419 in Home & Kitchen > Kitchen & Dining > Tabletop > Glassware & Drinkware, >#367,106 in Home & Kitchen > Kitchen & Dining > Food Service Equipment & Supplies]",Amazon Home,$12.48,0006564224,,[],[]
3,"[Home & Kitchen, Bath, Bathroom Accessories]",,"[The only soap in the world with a unique combination of glycerine and 100% pure, natural sandalwood oil. Mild, gentle Mysore Sandal Soap turns your bath into a beauty treatment. It contains no harsh chemicals to cause allergies or irritation. It contains pure sandal oil from the forests of Karnataka, India, which supply Mysore Sandal oil, the best in the world. Sandal Oil soothes dry skin and relieves rashes, bringing a glow to the face and leaving an exquisite lingering fragrance. Used the...",4 BARS! Mysore Sandal Soap 70grams FAST SHIPPING,,Mysore,[],"[>#6,942,841 in Home & Kitchen (See Top 100 in Home & Kitchen), >#212,123 in Home & Kitchen > Bath > Bathroom Accessories]",Amazon Home,$22.00,0009046461,,"[https://images-na.ssl-images-amazon.com/images/I/51q96B4YgyL._SS40_.jpg, https://images-na.ssl-images-amazon.com/images/I/51q96B4YgyL._SS40_.jpg, https://images-na.ssl-images-amazon.com/images/I/51q96B4YgyL._SS40_.jpg]","[https://images-na.ssl-images-amazon.com/images/I/51q96B4YgyL.jpg, https://images-na.ssl-images-amazon.com/images/I/51q96B4YgyL.jpg, https://images-na.ssl-images-amazon.com/images/I/51q96B4YgyL.jpg]"
4,"[Home & Kitchen, Home Dcor, Home Fragrance, Incense & Incense Holders, Incense]",,"[Divya Arogya vati improves health, It has natural anti-bio-tic and anti-viral properties, supports skin, imbalance of tridosh. It helps strengthening the immune system.This is a special combination of all the three .]",AROGYA VATI (40gm) by popeye seller,,Patanjali,[],"[>#3,103,399 in Home & Kitchen (See Top 100 in Home & Kitchen), >#7,765 in Home & Kitchen > Home Dcor > Home Fragrance > Incense & Incense Holders > Incense]",Amazon Home,$5.10,0234937912,,[],[]


In [10]:
# Calculate memory usage of the updated df
usage = df.memory_usage(deep=True) / (1024 ** 2)

# Sort and display
usage_df = usage.sort_values(ascending=False).to_frame(name='MB')
print(usage_df)
print(usage_df.sum())

                         MB
tech1            335.494279
title            152.646522
feature          137.998505
category         134.750366
rank             117.716008
description      116.782104
imageURL         106.388367
imageURLHighRes  106.388367
price             97.420394
main_cat          77.342453
date              75.461198
brand             74.584199
asin              73.177204
tech2             60.810126
Index              0.000126
MB    1666.960217
dtype: float64
